# 1. Análise Exploratória dos Dados (EDA)

In [ ]:
# Importação das bibliotecas e do DataFrame
import pandas as pd

df = pd.read_csv("../data/raw_data.csv")

df.head()

,ID_Pedido,Data_Compra,Cliente_ID,Produto,Categoria,Quantidade,Preco_Unitario,Status_Entrega
0,1001,2026-06-25,109.0,Smartwatch,Eletrônicos,1.0,5999.9,Cancelado
1,1002,2026-06-13,135.0,Smartwatch,Eletrônicos,3.0,8500.0,Entregue
2,1003,2026-06-05,113.0,Smartwatch,Acessórios,3.0,799.5,Entregue
3,1004,2026-06-20,130.0,Smartwatch,Acessórios,4.0,2100.0,Entregue
4,1005,2026-06-25,147.0,Smartwatch,Acessórios,4.0,850.0,Cancelado


In [3]:
# Verificação de informações gerais do DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Pedido       103 non-null    int64  
 1   Data_Compra     103 non-null    str    
 2   Cliente_ID      102 non-null    float64
 3   Produto         103 non-null    str    
 4   Categoria       103 non-null    str    
 5   Quantidade      97 non-null     float64
 6   Preco_Unitario  103 non-null    str    
 7   Status_Entrega  100 non-null    str    
dtypes: float64(2), int64(1), str(5)
memory usage: 6.6 KB


## Observações:
- Coluna 'Data_Compra' está no formato de string ao invés de datetime.
- Coluna 'Preco_Unitario' está no formato de string ao invés de de float64.
- Coluna 'Cliente_ID' está no formato float64 ao invés de Int64.

In [4]:
# Verificação de valores ausentes no DataFrame
df.isna().sum()

ID_Pedido         0
Data_Compra       0
Cliente_ID        1
Produto           0
Categoria         0
Quantidade        6
Preco_Unitario    0
Status_Entrega    3
dtype: int64

In [5]:
# Verificação de registros duplicados no DataFrame
print(f"Número de linhas duplicadas: {df.duplicated().sum()}")

Número de linhas duplicadas: 3


In [6]:
# Estátisticas descritivas para colunas númericas (Preço_Unitário não aparece)
df.describe()

,ID_Pedido,Cliente_ID,Quantidade
count,103.000000,102.000000,97.000000
mean,1049.087379,124.313725,3.103093
std,29.733821,14.072641,4.954973
min,1001.000000,100.000000,1.000000
25%,1023.500000,113.000000,2.000000
50%,1049.000000,125.000000,3.000000
75%,1074.500000,136.000000,4.000000
max,1100.000000,148.000000,50.000000


# 2. Limpeza e Pré-Processamento de Dados

In [7]:
# Criando cópia do DataFrame para manter original intacto
df_clean = df.copy()

In [8]:
# Correção do tipo de dados

# Correção de 'Preco_Unitario'
df_clean['Preco_Unitario'] = pd.to_numeric(df_clean['Preco_Unitario'], errors='coerce')

# Correção de 'Cliente_ID'
df_clean['Cliente_ID'] = pd.to_numeric(df_clean['Cliente_ID'], errors='coerce').astype('Int64')

# Correção de 'Data_Compra' 
df_clean['Data_Compra'] = pd.to_datetime(df_clean['Data_Compra'], errors='coerce')

# Verificação das trocas
df_clean.dtypes

ID_Pedido                  int64
Data_Compra       datetime64[us]
Cliente_ID                 Int64
Produto                      str
Categoria                    str
Quantidade               float64
Preco_Unitario           float64
Status_Entrega               str
dtype: object

In [9]:
# Remoção/Substituição de valores ausentes (NaN)
# Na tabela de quantidade, iremos preencher os valores ausentes com o valor da mediana, sendo mais robusta para outliers
qtd_mediana = df_clean['Quantidade'].median()
df_clean.fillna({'Quantidade': qtd_mediana}, inplace=True)
print(f"Valores ausentes tratados com sucesso! Tabela: {df.Quantidade.name}")

# Na tabela 'Status_Entrega', iremos preencher os valores ausentes com a moda (mais frequente)
moda_status_entrega = df_clean['Status_Entrega'].mode()[0]
df_clean.fillna({'Status_Entrega': moda_status_entrega}, inplace=True)
print(f"Valores ausentes tratados com sucesso! Tabela: {df.Status_Entrega.name}")  

# Na tabela 'Cliente_ID' e 'Preco_Unitario', a melhor abordagem para tratar 
# os valores ausentes é realizando a remoção, já que não é possível inferir esses dados
df_clean.dropna(subset= ['Cliente_ID', 'Preco_Unitario'], inplace=True)
print(f"Valores ausentes tratados com sucesso! Tabela: {df.Preco_Unitario.name} e {df.Cliente_ID.name}")

Valores ausentes tratados com sucesso! Tabela: Quantidade
Valores ausentes tratados com sucesso! Tabela: Status_Entrega
Valores ausentes tratados com sucesso! Tabela: Preco_Unitario e Cliente_ID


In [10]:
# Remoção de duplicatas
df_clean.drop_duplicates(inplace=True)
print(f"Registros duplicados removidos com sucesso!")

Registros duplicados removidos com sucesso!


In [ ]:
# Remoção da valores em 'Quantidade' que estão muito distantes da média.
# Uma abordagem comum é remover valores que estão além de 3 desvios padrão da média.
limite_superior = df_clean['Quantidade'].mean() + 3 * df_clean['Quantidade'].std()
df_clean = df_clean[df_clean['Quantidade'] < limite_superior]

In [12]:
# Verificação Final
print("\nVerificação Final Pós-Limpeza\n")
df_clean.info()
print("\nValores ausentes restantes:\n", df_clean.isna().sum())
print(f"\nLinhas duplicadas restantes: {df_clean.duplicated().sum()}")
df_clean.head()


Verificação Final Pós-Limpeza

<class 'pandas.DataFrame'>
Index: 98 entries, 0 to 99
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID_Pedido       98 non-null     int64         
 1   Data_Compra     98 non-null     datetime64[us]
 2   Cliente_ID      98 non-null     Int64         
 3   Produto         98 non-null     str           
 4   Categoria       98 non-null     str           
 5   Quantidade      98 non-null     float64       
 6   Preco_Unitario  98 non-null     float64       
 7   Status_Entrega  98 non-null     str           
dtypes: Int64(1), datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 7.0 KB

Valores ausentes restantes:
 ID_Pedido         0
Data_Compra       0
Cliente_ID        0
Produto           0
Categoria         0
Quantidade        0
Preco_Unitario    0
Status_Entrega    0
dtype: int64

Linhas duplicadas restantes: 0


,ID_Pedido,Data_Compra,Cliente_ID,Produto,Categoria,Quantidade,Preco_Unitario,Status_Entrega
0,1001,2026-06-25,109,Smartwatch,Eletrônicos,1.0,5999.9,Cancelado
1,1002,2026-06-13,135,Smartwatch,Eletrônicos,3.0,8500.0,Entregue
2,1003,2026-06-05,113,Smartwatch,Acessórios,3.0,799.5,Entregue
3,1004,2026-06-20,130,Smartwatch,Acessórios,4.0,2100.0,Entregue
4,1005,2026-06-25,147,Smartwatch,Acessórios,4.0,850.0,Cancelado
